# 14 — RQ3: sinh Candidate Requirement (máy local)

**Phạm vi**

Sinh Candidate Requirement (User Story tiếng Anh, Acceptance Criteria, Evidence IDs) cho các issue group đủ điều kiện. Đầu vào là kết
quả gom cụm đã khóa (amendment v2.4) và trạng thái temporal (v2.5: cả 31 group đủ điều kiện đều RECURRING). Việc chọn cụm và lọc
evidence theo amendment v2.6.

**Bốn bước của amendment v2.6**

(1) Cụm manual purity < 0,25 (9, 2, 13, 4): không sinh, liệt kê cho BA xem thủ công. (2) Cụm 0 và 8: sàng lọc từng review, chỉ review
nêu lỗi hoặc đề xuất chức năng app cụ thể được làm evidence. (3) Cụm 0,25 ≤ purity < 0,60 (11, 12, 14; cụm mixed 7 theo Q14): evidence
chỉ gồm review đúng loại. (4) Các cụm còn lại (1, 3, 5, 6, 10, 15, 16): sinh bình thường.

**Mô hình**

`gpt-4o-mini-2024-07-18`, temperature 0, seed 42; prompt khóa và ghi hash trong `04_prompts/prompts_registry.md`. Log từng lượt gọi không
chứa văn bản review.

**Hai phần, có điểm dừng**

Phần 1 sàng lọc cụm 0 và 8 rồi xuất phiếu QA; phần 2 (sinh requirement) chỉ chạy sau khi người nghiên cứu QA kết quả sàng lọc.

In [ ]:
import os, sys, json, hashlib, datetime, subprocess
import numpy as np, pandas as pd, yaml
ROOT = os.path.abspath('..')
sys.path.insert(0, os.path.join(ROOT, '02_src'))
from src.utils.seeds import set_seed
set_seed(42)
from src.utils.display import show as show_table
from src.utils.llm import check_api, get_client, prices
from src.generation import screening as SC
RQ2 = os.path.join(ROOT, '05_experiments', 'rq2_topics')
RQ3 = os.path.join(ROOT, '05_experiments', 'rq3_requirements')
QA_DIR = os.path.join(ROOT, '07_human_eval', 'scope_screening')
os.makedirs(RQ3, exist_ok=True); os.makedirs(QA_DIR, exist_ok=True)
COMMIT = subprocess.run(['git', '-C', ROOT, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
print('Ngày chạy:', datetime.date.today().strftime('%d/%m/%Y'), '| commit:', COMMIT)

## Dữ liệu và đối chiếu

**Kết quả đã khóa**

`topics.csv` và `review_topic_map.csv` phải trùng sha256 của amendment v2.4; trạng thái temporal đọc từ `gd13_temporal_groups.csv`.

In [ ]:
FZ12 = yaml.safe_load(open(os.path.join(ROOT, '03_configs', 'frozen', 'rq2_topics_config.yaml'), encoding='utf-8'))['rq2_topics']
for f in ('topics.csv', 'review_topic_map.csv'):
    assert hashlib.sha256(open(os.path.join(RQ2, f), 'rb').read()).hexdigest() == FZ12['locked_result']['files_sha256'][f], f
TOPICS = pd.read_csv(os.path.join(RQ2, 'topics.csv'), encoding='utf-8-sig')
MAP = pd.read_csv(os.path.join(RQ2, 'review_topic_map.csv'))
ACT = pd.read_csv(os.path.join(RQ2, 'actionable_reviews.csv'), encoding='utf-8-sig')
TEMP = pd.read_csv(os.path.join(RQ2, 'gd13_temporal_groups.csv'))
print('Group đủ điều kiện temporal:', int(TEMP['eligible_for_generation'].sum()), '/', len(TEMP),
      '| trạng thái:', TEMP['status'].value_counts().to_dict())
SCOPE_TOPICS = [0, 8]
SCR_IN = MAP[MAP['topic_id'].isin(SCOPE_TOPICS)][['review_id', 'topic_id']].merge(ACT[['review_id', 'text']], on='review_id')
print('Review cần sàng lọc:', SCR_IN.groupby('topic_id').size().to_dict(), '| tổng', len(SCR_IN))

## Phần 1 — sàng lọc phạm vi cụm 0 và 8 (bước 2 của v2.6)

**Prompt**

`04_prompts/requirement_generation/scope_screening_system_v2.txt` (hash trong registry; khác thì dừng). Nhãn `app_actionable`: review nêu
ít nhất một lỗi hoặc đề xuất cụ thể về chức năng, giao diện, luồng thao tác, hiệu năng hay dữ liệu hiển thị của ứng dụng; nhãn
`out_of_scope`: chỉ gồm vận hành giao hàng, shipper, CSKH, yêu cầu kiểm soát hay chế tài người bán, chất lượng và giá sản phẩm, khen
hoặc than phiền chung. Review có cả hai loại ý được tính `app_actionable`. Ví dụ trong prompt là câu viết lại, không lấy từ cụm 0, 8.

**Chạy thử và prompt v2**

10 review ngẫu nhiên mỗi cụm (seed 42) để kiểm tra cách parse và chi phí thật. Chạy thử với v1 cho thấy LLM tính cả việc shipper
không đến lấy hàng hay nhân viên không hướng dẫn là lỗi ứng dụng, và có lý do tự suy ra; v2 thêm quy tắc "việc một người làm hay không
làm điều gì không phải lỗi ứng dụng" và yêu cầu lý do bám đúng lời review. Cùng 20 review được chạy lại với v2 để so; chạy toàn bộ
chỉ khi `CONFIRM_SCREENING = True`.

In [ ]:
r = check_api()
assert r['ok'] and r['snapshot_match'], {k: v for k, v in r.items() if k != 'detail'}
client, SETTINGS = get_client()
PRICES = prices()
PROMPT_VERSION = 'v2'
SYSTEM, PHASH = SC.load_prompt(PROMPT_VERSION)
assert SC.registered(PHASH), 'prompt sàng lọc chưa có trong prompts_registry.md'
print('Model:', r['response_model'], '| prompt hash:', PHASH[:16], '| giá:', PRICES)

PILOT_PATH = os.path.join(RQ3, f'gd14_scope_screening_pilot_{PROMPT_VERSION}.csv')
sample = pd.concat([g.sample(10, random_state=42) for _, g in SCR_IN.groupby('topic_id')])
if os.path.exists(PILOT_PATH):
    PILOT = pd.read_csv(PILOT_PATH)
else:
    PILOT = SC.screen(sample, SYSTEM, client, SETTINGS, os.path.join(RQ3, f'gd14_scope_screening_pilot_{PROMPT_VERSION}_calls.jsonl'))
    PILOT = PILOT.merge(sample[['review_id', 'topic_id']], on='review_id')
    PILOT.to_csv(PILOT_PATH, index=False)
V1_PATH = os.path.join(RQ3, 'gd14_scope_screening_pilot_v1.csv')
if PROMPT_VERSION != 'v1' and os.path.exists(V1_PATH):
    cmp = PILOT[['review_id', 'topic_id', 'label', 'reason']].merge(pd.read_csv(V1_PATH)[['review_id', 'label']].rename(columns={'label': 'label_v1'}), on='review_id')
    print('Chạy thử: đổi nhãn so với v1:', int((cmp['label'] != cmp['label_v1']).sum()), '/', len(cmp))
view = PILOT.merge(SCR_IN[['review_id', 'text']], on='review_id')
view['text'] = view['text'].astype(str).str.slice(0, 140)
show_table(view[['topic_id', 'label', 'reason', 'text']], 'Chạy thử sàng lọc (10 review mỗi cụm)', wrap=['text', 'reason'])
c = SC.cost(PILOT, PRICES)
print('Chạy thử:', c)
print(f"Ước lượng cho {len(SCR_IN)} review: ~${c['chi_phí_usd'] / len(PILOT) * len(SCR_IN):.3f}, ~{c['giây'] / len(PILOT) * len(SCR_IN) / 60:.0f} phút")

## Sàng lọc toàn bộ

**Điều kiện**

Chỉ chạy khi `CONFIRM_SCREENING = True`, sau khi kết quả chạy thử đã được xem; nếu file kết quả đã có thì không gọi API lại.

**Kết quả và phiếu QA**

`05_experiments/rq3_requirements/gd14_scope_screening.csv` (không có văn bản review). Phiếu QA
`07_human_eval/scope_screening/scope_screening_qa.csv`: mỗi cụm 20 review — 10 review mỗi nhãn, rút ngẫu nhiên (seed 42), xáo thứ tự — có
văn bản, nhãn và lý do của LLM, cột `qa_label` để người nghiên cứu điền; không commit.

In [ ]:
CONFIRM_SCREENING = True          # đã xác nhận ngày 02/11/2026 sau chạy thử (prompt v2)
FULL_PATH = os.path.join(RQ3, 'gd14_scope_screening.csv')
if os.path.exists(FULL_PATH):
    SCR = pd.read_csv(FULL_PATH)
elif CONFIRM_SCREENING:
    SCR = SC.screen(SCR_IN, SYSTEM, client, SETTINGS, os.path.join(RQ3, 'gd14_scope_screening_calls.jsonl'))
    SCR = SCR.merge(SCR_IN[['review_id', 'topic_id']], on='review_id').assign(prompt_version=PROMPT_VERSION, prompt_hash=PHASH)
    SCR.to_csv(FULL_PATH, index=False)
else:
    SCR = None
    print('Chưa sàng lọc toàn bộ (CONFIRM_SCREENING = False).')
if SCR is not None:
    summ = SCR.groupby('topic_id')['label'].value_counts().unstack(fill_value=0)
    summ['tỷ lệ app_actionable'] = summ.get('app_actionable', 0) / summ.sum(axis=1)
    show_table(summ.reset_index(), 'Kết quả sàng lọc theo cụm', percent=['tỷ lệ app_actionable'])
    print('Chi phí thật:', SC.cost(SCR, PRICES))
    QA_PATH = os.path.join(QA_DIR, 'scope_screening_qa.csv')
    if not os.path.exists(QA_PATH):
        SC.qa_sheet(SCR, ACT, TOPICS).to_csv(QA_PATH, index=False, encoding='utf-8-sig')
        print('Đã tạo phiếu QA:', os.path.relpath(QA_PATH, ROOT))
    else:
        print('Phiếu QA đã có, không ghi đè:', os.path.relpath(QA_PATH, ROOT))

## Kết quả QA sàng lọc

**Cách tính**

Phiếu `07_human_eval/scope_screening/scope_screening_qa.csv` do người nghiên cứu chấm (`qa_label`). Mẫu phân tầng theo nhãn LLM (10 review
mỗi nhãn mỗi cụm), nên tỷ lệ đồng ý trong từng nhãn là precision của nhãn đó; accuracy và recall `app_actionable` ở mức cụm được ước tính
bằng cách gán trọng số theo số review LLM đã xếp vào mỗi nhãn. Kết quả lưu `gd14_scope_screening_qa_summary.csv` (không có văn bản review).

In [ ]:
QA = pd.read_csv(os.path.join(QA_DIR, 'scope_screening_qa.csv'), encoding='utf-8-sig')
assert QA['qa_label'].notna().all(), 'phiếu QA chưa chấm xong'
QA_SUMMARY = SC.qa_score(QA, SCR)
QA_SUMMARY.to_csv(os.path.join(RQ3, 'gd14_scope_screening_qa_summary.csv'), index=False)
show_table(QA_SUMMARY.drop(columns=['est_true_app_actionable']), 'Độ chính xác của bước sàng lọc so với QA của người nghiên cứu',
           decimals=2)
dis = QA[QA['llm_label'] != QA['qa_label']][['item_no', 'topic_id', 'llm_label', 'qa_label', 'qa_note']]
show_table(dis, 'Các review LLM và người nghiên cứu gán khác nhau', wrap=['qa_note'])

**Nhận xét**

Bước sàng lọc đồng ý với người nghiên cứu ở 35/40 review (87,5%, κ = 0,75). Nhãn `app_actionable` đúng 9/10 ở cụm 0 và 8/10 ở cụm 8,
nên review được giữ làm evidence phần lớn đúng là lỗi hoặc đề xuất chức năng app; hai ca bị giữ nhầm ở cụm 8 là đề xuất còn chung chung
(điều kiện hưởng voucher, "cải thiện khi mua nhiều món"), một ca ở cụm 0 là yêu cầu bỏ ghép đơn thuộc điều phối giao hàng. Sai số chính là
bỏ sót ở cụm 0: 2/10 review LLM xếp `out_of_scope` thực chất là chức năng app (thiếu thông tin liên hệ shipper; trạng thái "giao thành công"
hiển thị khi chưa giao), nên ước tính chỉ khoảng 64% review app thật của cụm 0 được giữ; ở cụm 8 không có ca bỏ sót trong mẫu. Prompt v2 được
giữ nguyên (amendment v2.7): bỏ sót làm tần suất issue của cụm 0 bị đếm thấp nhưng không đưa review ngoài phạm vi vào evidence, và chỉnh
prompt trên chính mẫu QA sẽ làm mẫu QA mất giá trị đánh giá. Mẫu nhỏ nên các tỷ lệ trên có khoảng tin cậy rộng.

## Phần 2 — sinh requirement

**Unit sinh requirement**

Mỗi unit là một cụm, hoặc một cặp (cụm × loại) khi tập evidence là mixed (Q14). Quy tắc theo amendment v2.6 và v2.7, cấu hình trong
`03_configs/frozen/rq3_generation_config.yaml` (khóa theo v2.8): cụm 9, 2, 13, 4 chuyển BA; cụm 0 và 8 chỉ giữ review `app_actionable`, rồi áp lại luật loại
chủ đạo ≥ 60% trên tập đã lọc; cụm 11, 12, 14 chỉ giữ review đúng loại chủ đạo; cụm 7 tách theo loại; các cụm còn lại dùng toàn bộ review.
Luật temporal v2.5 được tính lại trên đúng tập evidence của từng unit; chỉ ACTIVE và RECURRING được sinh.

**Truy vết đầy đủ**

Mỗi review actionable có đúng một unit hoặc một lý do loại (`ba_manual`, `out_of_scope_app`, `type_mismatch`, `mixed_origin_below_q14`,
`outlier`), lưu ở `gd14_evidence_pool.csv`; danh sách cho BA ở `gd14_ba_manual_review.csv` (kèm review_id, không có văn bản).

In [ ]:
from src.generation import core as G
from src.temporal import core as TC
GCFG = G.load_config(ROOT, 'frozen')
TCFG = TC.load_config(ROOT)
UNITS, POOL = G.build_units(GCFG, TOPICS, MAP, SCR)
UNITS = G.unit_temporal(UNITS, POOL, ACT, TCFG, TC)
POOL['used_for_generation'] = POOL['unit_id'].map(UNITS.set_index('unit_id')['eligible']).fillna(False).astype(bool)
show_table(UNITS[['unit_id', 'step', 'type', 'n_pool', 'topic_size', 'issue_frequency', 'N', 'c30', 'R90', 'n_win', 'temporal_status',
                  'mixed_origin_flag']], 'Unit sinh requirement', percent=['issue_frequency'], decimals=2)
state = POOL['exclusion_reason'].where(POOL['exclusion_reason'].notna(),
                                       np.where(POOL['used_for_generation'], 'evidence', 'temporal_không_đủ'))
show_table(state.value_counts().rename_axis('trạng thái review').reset_index(name='số review'), 'Phân bổ review actionable')
PURITY = pd.read_csv(os.path.join(RQ2, 'gd12_manual_purity.csv'), encoding='utf-8-sig')
BA = G.ba_manual_table(POOL, TOPICS, PURITY)
show_table(BA.groupby(['topic_id', 'name_vi', 'size', 'manual_purity']).size().reset_index(name='số review_id'),
           'Cụm cần BA xem thủ công (v2.6 bước 1)', decimals=2)
UNITS.to_csv(os.path.join(RQ3, 'gd14_generation_units.csv'), index=False, encoding='utf-8-sig')
POOL.to_csv(os.path.join(RQ3, 'gd14_evidence_pool.csv'), index=False)
BA.to_csv(os.path.join(RQ3, 'gd14_ba_manual_review.csv'), index=False, encoding='utf-8-sig')

**Prompt và evidence đưa vào prompt**

`04_prompts/requirement_generation/requirement_generation_system_<version>.txt` (version trong config, hash trong registry; khác thì dừng), tiếng Anh, có template tên
theo loại, dạng User Story "As a …, I want …, so that …." và 2–3 Acceptance Criteria Given/When/Then. Mỗi unit đưa vào prompt 20 review gần
tâm tập evidence nhất (cosine trên embedding mpnet đã cache ở GĐ12; ưu tiên review gần tâm theo giới hạn (a) của v2.6), nhãn E1..E20;
JSON schema chỉ cho phép các nhãn này làm evidence. Backend kiểm tra từng câu trả lời (số requirement, tiền tố tên, dạng User Story, số
và dạng AC, 3–10 evidence trong danh sách); vi phạm thì gọi lại một lần kèm danh sách vi phạm. Log từng lượt gọi chỉ ghi review_id đã
đưa vào prompt, không ghi văn bản review.

In [ ]:
GSYSTEM, GHASH, GSCHEMA = G.load_prompt(GCFG['prompt_version'])
assert SC.registered(GHASH), 'prompt sinh requirement chưa có trong prompts_registry.md'
assert GHASH == GCFG['prompt_hash'], 'prompt khác bản khóa trong config frozen'
print('Prompt sinh requirement:', GCFG['prompt_version'], '| hash:', GHASH[:16])
EMB = np.load(os.path.join(RQ2, 'cache', 'mpnet_actionable.npy'))
meta = json.load(open(os.path.join(RQ2, 'cache', 'mpnet_actionable.json'), encoding='utf-8'))
assert meta['n'] == len(ACT) == len(EMB) and meta['revision'] == FZ12['embedding']['revision']
ACT_INDEX = {r: i for i, r in enumerate(ACT['review_id'])}
TEXT = ACT.set_index('review_id')['text']
TOPIC_ROW = TOPICS.set_index('topic_id')
ELIGIBLE = UNITS[UNITS['eligible']].reset_index(drop=True)
PAYLOADS = {}
for _, u in ELIGIBLE.iterrows():
    shown = G.shown_evidence(POOL, u['unit_id'], EMB, ACT_INDEX, GCFG['evidence']['k_shown'])
    PAYLOADS[u['unit_id']] = (G.user_message(u, TOPIC_ROW.loc[u['topic_id']], shown, TEXT, GCFG['evidence']['max_chars']), shown)
print('Unit được sinh:', len(ELIGIBLE), '| không đủ điều kiện temporal:', UNITS.loc[~UNITS['eligible'], 'unit_id'].tolist())

**Chạy thử và prompt v2**

Ba unit đại diện cho ba cách lọc: cụm 1 (bình thường, BUG), 7-UX (cụm mixed tách theo loại), 8 (lọc theo nội dung chức năng app). Chạy thử
với v1 qua mọi kiểm tra định dạng và evidence, nhưng đọc review được trích thì requirement của cụm 8 ("chọn đơn vị giao hàng") trích các review
nói về vấn đề khác (kiểm hàng khi nhận, vị trí shop, theo dõi vận chuyển), và AC của 7-UX thêm ngữ cảnh review không nêu. V2 đổi thứ tự sinh:
chọn evidence cùng một vấn đề trước, viết `problem_vi` (vấn đề chung bằng tiếng Việt), rồi mới viết User Story và AC; AC chỉ kiểm tra tình huống
có trong review được trích. Chạy lại 3 unit với v2: cụm 1 tốt hơn, nhưng cụm 8 vẫn trích review khác vấn đề (luật "≥ 3 evidence" buộc trích
thêm khi cụm gồm nhiều đề xuất nhỏ), AC hiểu "xem hàng" thành xem trang sản phẩm, và 7-UX hiểu quảng cáo hiện trong game/TikTok thành chuyển
hướng ngay trong app. V3: tối thiểu 2 evidence, chỉ trích review nói đúng vấn đề; nói rõ review có thể than quảng cáo hay link của app hiện trong
app khác; luật chống bịa — User Story và AC chỉ dùng thông tin có trong review được trích, review chưa đủ rõ thì viết AC tổng quát hơn. V3 chạy
thử trên 5 unit (thêm cụm 3 về quảng cáo và 0-FEATURE). Kết quả lưu `gd14_generation_pilot_<version>.json`; nếu file đã có thì không gọi API
lại.

In [ ]:
PILOT_UNITS = {'v1': ['1', '7-UX', '8'], 'v2': ['1', '7-UX', '8'], 'v3': ['1', '3', '0-FEATURE', '7-UX', '8']}

def run_generation(units, path, log_path):
    if os.path.exists(path):
        g = pd.read_json(path, orient='records', dtype={'unit_id': str})
        return g
    g = G.generate(units, GSYSTEM, GSCHEMA, client, SETTINGS, GCFG, PAYLOADS, log_path)
    g.assign(prompt_version=GCFG['prompt_version'], prompt_hash=GHASH).to_json(path, orient='records', force_ascii=False, indent=1)
    return g

def show_requirements(reqs, caption):
    v = reqs.assign(acceptance_criteria=reqs['acceptance_criteria'].map(lambda s: '\n'.join(json.loads(s))))
    cols = ['requirement_id', 'type', 'title_vi', 'problem_vi', 'user_story', 'acceptance_criteria', 'n_evidence_cited', 'validation_issues']
    show_table(v[[c for c in cols if c in v and (c != 'problem_vi' or v[c].astype(bool).any())]], caption,
               wrap=['title_vi', 'problem_vi', 'user_story', 'acceptance_criteria'])

PV = GCFG['prompt_version']
PGEN = run_generation(ELIGIBLE[ELIGIBLE['unit_id'].isin(PILOT_UNITS[PV])], os.path.join(RQ3, f'gd14_generation_pilot_{PV}.json'),
                      os.path.join(RQ3, f'gd14_generation_pilot_{PV}_calls.jsonl'))
PREQ, PEV = G.requirements_table(PGEN, UNITS, PAYLOADS)
for old in [v for v in PILOT_UNITS if v < PV]:
    f = os.path.join(RQ3, f'gd14_generation_pilot_{old}.json')
    if os.path.exists(f):
        OREQ, _ = G.requirements_table(pd.read_json(f, orient='records', dtype={'unit_id': str}), UNITS, PAYLOADS)
        show_requirements(OREQ, f'Chạy thử với prompt {old} (để so)')
show_requirements(PREQ, f'Chạy thử sinh requirement ({len(PGEN)} unit, prompt {PV})')
chk, ok = G.verify_evidence(PREQ, PEV, POOL, ACT, MAP, UNITS, PAYLOADS)
show_table(chk, 'Kiểm tra evidence (chạy thử)')
pc = G.cost(PGEN, PRICES)
print('Chạy thử:', pc)
print(f"Ước lượng cho {len(ELIGIBLE)} unit: ~${pc['chi_phí_usd'] / len(PGEN) * len(ELIGIBLE):.3f}")

**Evidence của chạy thử để đối chiếu**

Văn bản các review được trích dẫn (prompt đang dùng), để người nghiên cứu kiểm tra requirement có bám review hay không (chỉ hiển thị,
không lưu).

In [ ]:
v = PEV.copy()
v['text'] = v['review_id'].map(TEXT).astype(str).str.slice(0, 200)
show_table(v[['requirement_id', 'evidence_label', 'text']], 'Review được trích dẫn trong chạy thử', wrap=['text'])

## Sinh requirement toàn bộ

**Điều kiện**

Chỉ chạy khi `CONFIRM_GENERATION = True`, sau khi kết quả chạy thử đã được xem; cấu hình dev đã được chép sang
`03_configs/frozen/rq3_generation_config.yaml` và prompt v3 là bản cuối (amendment v2.8). Nếu file kết quả đã có thì không gọi API lại.

**Đọc kiểm chạy thử v3**

Evidence trích đúng vấn đề khoảng 18/25 review (cụm 3: 5/5; 7-UX: 5/6; cụm 1: 4/5; 0-FEATURE: 2–3/5; cụm 8: 2/4); 6/15 AC bịa ngữ cảnh hoặc lệch
vấn đề (cụm 8: cả 3 AC hiểu "xem hàng" thành xem trang sản phẩm). Luật về quảng cáo hiện trong app khác có hiệu quả (7-UX đúng bối cảnh),
còn luật chống bịa chưa được model tuân thủ đầy đủ. Không thêm bước LLM kiểm tra evidence; fidelity và hallucination được đo bằng người ở
GĐ15, và output là Candidate Requirement cần BA duyệt (amendment v2.8).

In [ ]:
CONFIRM_GENERATION = True          # đã xác nhận ngày 02/11/2026 sau chạy thử 5 unit (prompt v3, amendment v2.8)
GEN_PATH = os.path.join(RQ3, 'gd14_generation.json')
if CONFIRM_GENERATION or os.path.exists(GEN_PATH):
    GEN = run_generation(ELIGIBLE, GEN_PATH, os.path.join(RQ3, 'gd14_generation_calls.jsonl'))
else:
    GEN = None
    print('Chưa sinh toàn bộ (CONFIRM_GENERATION = False).')

## Kiểm tra evidence và structured output

**Kiểm tra ở backend**

Mỗi evidence ID phải tồn tại trong corpus actionable, thuộc đúng cụm của requirement, thuộc tập evidence của unit và nằm trong danh sách đưa
vào prompt (protocol RQ3); mỗi requirement phải có ít nhất một evidence. Không đạt thì dừng.

**File xuất (không có văn bản review)**

`requirements.csv` (requirement_id, cluster_id, type, user_story, acceptance_criteria, evidence_ids, temporal_status, mixed_origin_flag,
scope_note, cùng title_vi, problem_vi, unit_id, issue_frequency, số evidence) và `requirement_evidence.csv` (một dòng mỗi review được trích) trong
`05_experiments/rq3_requirements/`. Output không được chỉnh tay trước đánh giá (GĐ15).

In [ ]:
REQ, REV = G.requirements_table(GEN, UNITS, PAYLOADS)
CHECK, OK = G.verify_evidence(REQ, REV, POOL, ACT, MAP, UNITS, PAYLOADS)
show_table(CHECK, 'Kiểm tra evidence ID ở backend')
assert OK, 'evidence ID không hợp lệ — dừng'
REQ = REQ.sort_values('issue_frequency', ascending=False, kind='stable').reset_index(drop=True)
REQ.to_csv(os.path.join(RQ3, 'requirements.csv'), index=False, encoding='utf-8-sig')
REV.to_csv(os.path.join(RQ3, 'requirement_evidence.csv'), index=False)
show_requirements(REQ, f'Candidate Requirement ({len(REQ)} requirement, {REQ["unit_id"].nunique()} unit), xếp theo issue frequency')
show_table(REQ.groupby('type').agg(số_requirement=('requirement_id', 'size'), số_unit=('unit_id', 'nunique'),
                                   evidence_trung_bình=('n_evidence_cited', 'mean')).reset_index(),
           'Số requirement theo loại', decimals=1)

**Cụm cần BA xem thủ công**

Cụm 9, 2, 13, 4 (v2.6 bước 1) không sinh requirement tự động; danh sách review_id ở `gd14_ba_manual_review.csv`.

In [ ]:
show_table(BA.groupby(['topic_id', 'name_vi', 'size', 'manual_purity']).agg(
    review_ids_đầu=('review_id', lambda s: ', '.join(s.head(3)) + ', …')).reset_index().assign(ghi_chú='cần BA xem thủ công'),
    'Cụm chuyển BA (review_id đầy đủ trong gd14_ba_manual_review.csv)', decimals=2)

## Chi phí thật và thông tin lượt chạy

Token và chi phí lấy từ `usage` của API (giá đọc từ `.env`); gồm sàng lọc, các lần chạy thử và lượt sinh toàn bộ. Lưu `gd14_run_info.json`.

In [ ]:
rows = []
for name, f, kind in [('sàng lọc chạy thử v1', 'gd14_scope_screening_pilot_v1.csv', 'scr'),
                      ('sàng lọc chạy thử v2', 'gd14_scope_screening_pilot_v2.csv', 'scr'),
                      ('sàng lọc toàn bộ (v2)', 'gd14_scope_screening.csv', 'scr'),
                      ('sinh requirement chạy thử v1', 'gd14_generation_pilot_v1.json', 'gen'),
                      ('sinh requirement chạy thử v2', 'gd14_generation_pilot_v2.json', 'gen'),
                      ('sinh requirement chạy thử v3', 'gd14_generation_pilot_v3.json', 'gen'),
                      ('sinh requirement toàn bộ (v3)', 'gd14_generation.json', 'gen')]:
    d = pd.read_csv(os.path.join(RQ3, f)) if kind == 'scr' else pd.read_json(os.path.join(RQ3, f), orient='records', dtype={'unit_id': str})
    c = SC.cost(d, PRICES) if kind == 'scr' else G.cost(d, PRICES)
    rows.append({'bước': name, 'số lượt': len(d), 'token vào': c['token_vào'], 'token ra': c['token_ra'],
                 'chi phí (USD)': c['chi_phí_usd'], 'thời gian (giây)': c['giây']})
COST = pd.DataFrame(rows)
COST = pd.concat([COST, pd.DataFrame([{'bước': 'tổng', **COST.drop(columns='bước').sum().to_dict()}])], ignore_index=True)
show_table(COST, 'Chi phí thật của GĐ14', decimals=4)
import importlib.metadata as md_
INFO = {'run_date': datetime.date.today().isoformat(), 'commit': COMMIT, 'model': SETTINGS.snapshot,
        'temperature': SETTINGS.temperature, 'seed': G.API_SEED, 'response_models': sorted(GEN['model'].unique().tolist()),
        'screening_prompt': {'version': PROMPT_VERSION, 'hash': PHASH},
        'generation_prompt': {'version': GCFG['prompt_version'], 'hash': GHASH}, 'config': '03_configs/frozen/rq3_generation_config.yaml',
        'prices_per_1M': PRICES, 'n_units_generated': int(len(ELIGIBLE)), 'n_requirements': int(len(REQ)),
        'n_evidence_links': int(len(REV)), 'evidence_check_ok': OK, 'retries': int((GEN['attempts'] > 1).sum()),
        'units_with_violations': int(GEN['issues'].map(bool).sum()),
        'screening_qa': QA_SUMMARY.to_dict(orient='records'),
        'cost': COST.to_dict(orient='records'),
        'versions': {p: md_.version(p) for p in ('openai', 'pandas', 'numpy')}}
json.dump(INFO, open(os.path.join(RQ3, 'gd14_run_info.json'), 'w', encoding='utf-8'), ensure_ascii=False, indent=1, default=str)
print('Đã lưu:', sorted(f for f in os.listdir(RQ3) if not f.endswith('.jsonl')))